In [ ]:
# Relevant packages for Stanford data visualization
import open3d as o3d
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import time

In [ ]:
def plot_points(points, viewpoint, plot_viewpoint=True, 
                points_special_inds=None, extra_viewpoint=None, out_file="output.eps"):
    # Prepare point clouds
    if points_special_inds is not None:
        scatter_points_special = go.Scatter3d(
            x=points[points_special_inds, 0], 
            y=points[points_special_inds, 1], 
            z=points[points_special_inds, 2], 
            mode='markers',
            marker=dict(
                size=0.7,
                opacity=1,
                color='green'
            )
        )

        scatter_points_normal = go.Scatter3d(
            x=np.delete(points[:, 0], points_special_inds), 
            y=np.delete(points[:, 1], points_special_inds), 
            z=np.delete(points[:, 2], points_special_inds), 
            mode='markers',
            marker=dict(
                size=0.7,
                opacity=1,
                color='red'
            )
        )
    else:
        scatter_points_normal = go.Scatter3d(
            x=points[:, 0], 
            y=points[:, 1], 
            z=points[:, 2], 
            mode='markers',
            marker=dict(
                size=0.7,
                opacity=1,
                color='white'
            )
        )
    # Store point clouds
    data = [scatter_points_normal]

    if points_special_inds is not None:
        data.append(scatter_points_special)

    # Create a Scatter3d object for the viewpoint
    if plot_viewpoint:
        scatter_viewpoint = go.Scatter3d(
            x=[viewpoint[0]], 
            y=[viewpoint[1]], 
            z=[viewpoint[2]], 
            mode='markers',
            marker=dict(
                size=10,
                color='blue',
                opacity=1
            )
        )
        data.append(scatter_viewpoint)

    if extra_viewpoint is not None:
        scatter_extra_viewpoint = go.Scatter3d(
            x=[extra_viewpoint[0]], 
            y=[extra_viewpoint[1]], 
            z=[extra_viewpoint[2]], 
            mode='markers',
            marker=dict(
                size=10,
                color='yellow',
                opacity=1
            )
        )
        data.append(scatter_extra_viewpoint)
        
    fig = go.Figure(data=data)


    # Update layout to remove background and axes
    fig.update_layout(
        autosize=False,
        width=500,
        height=500,
        margin=dict(
            l=0,  # left margin
            r=0,  # right margin
            b=0,  # bottom margin
            t=0,  # top margin
            pad=0  # padding
        ),
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            camera=dict(
                up=dict(x=0, y=1, z=0),
                center=dict(x=0, y=0, z=0),
                eye=dict(x=viewpoint[0], y=viewpoint[1], z=viewpoint[2])  # Adjust the eye position here
            )
        ),
        showlegend=False,
        plot_bgcolor='rgba(0,0,0,0)',
        paper_bgcolor='rgba(0,0,0,0)',
    )
    # # Show the plot
    fig.show()

    #pio.write_image(fig, out_file)

In [ ]:
from utils.visibility import visible_points
# Read the ply file
pcd = o3d.io.read_point_cloud("data/stanford/bunny/reconstruction/bun_zipper.ply")
# Convert to numpy array
points = np.asarray(pcd.points)

viewpoint = np.array([0, 0, -1.55])
#viewpoint = np.array([0, 0, -0.3])
plot_points(points, viewpoint, plot_viewpoint=True) # out_file="images/point_clouds/stanford/bunny.eps")
# I set param_radius to 3.7 , now, change it later

t1 = time.perf_counter()
visible_inds = visible_points(points, viewpoint, param_radius=3.7)
print(f"time taken {time.perf_counter() - t1}")
plot_points(points[visible_inds], viewpoint, plot_viewpoint=True) #, out_file="images/point_clouds/stanford/bunny_visible_pc.eps")

In [ ]:
import nuscenes as ns

# Init Nusc object
data_folder = 'data/nuscenes/'
version = 'v1.0-mini'
nusc = ns.nuscenes.NuScenes(version=version, dataroot=data_folder, verbose=False)

In [ ]:
# Some visualization on visibility on the separate Nuscens point clouds

from utils.data_handling import read_nuscenes_data, split_data
from utils.visibility import visible_points
import torch
# Set params
N = 1
T_close_thresh = 1.5

#
PC_scenes = read_nuscenes_data(nusc, downsample_factor=2, n_scenes=N, n_samples=N, T_close_thresh=T_close_thresh)
PC_scenes_training, PC_scenes_test = split_data(PC_scenes, scenes_training=N)

for i in range(len(PC_scenes_training)):
    PC_pair = PC_scenes_training[i][0]
    zero_vec = np.zeros((3))
    T0 = PC_pair.pose0
    T1 = PC_pair.pose1
    viewpoint1_CS0 = torch.matmul(torch.linalg.inv(T0), T1).cpu().numpy()[:3, 3]
    pc_in = PC_pair.PC0.pc.cpu().numpy()

    # Remove close points
    pc_in = pc_in[np.linalg.norm(pc_in, axis=1) >= T_close_thresh]
    #
    print(f"\nScene: {i}")
    for radius in [3.7]:
        visible_inds = visible_points(pc_in, zero_vec, param_radius=radius)
        visible_inds_other = visible_points(pc_in, viewpoint1_CS0, param_radius=radius)
        print(f"Percentage visible points own viewpoint vs other " +
              f"{np.around(100*len(visible_inds)/pc_in.shape[0], 3)} % vs "+
              f"{np.around(100*len(visible_inds_other)/pc_in.shape[0], 3)} %")

    print(f"Percentage of point within {T_close_thresh} m {np.around(100*pc_in[np.linalg.norm(pc_in, axis=1) < T_close_thresh].shape[0]/pc_in.shape[0], 4)} %")
special_points = np.linalg.norm(pc_in, axis=1) < 2
#plot_points(pc_in, zero_vec, points_special_inds=visible_inds, extra_viewpoint=viewpoint1_CS0)
plot_points(pc_in, viewpoint=zero_vec, plot_viewpoint=False, points_special_inds=special_points)


In [ ]:
# Some visualization on visibility on the joint Nuscens point clouds

from utils.data_handling import read_nuscenes_data, split_data
from utils.visibility import covisible_inds
import torch
# Set params
N = 10
T_close_thresh = 1.5

#
PC_scenes = read_nuscenes_data(nusc, downsample_factor=1, n_scenes=N, n_samples=N, T_close_thresh=T_close_thresh)
PC_scenes_training, PC_scenes_test = split_data(PC_scenes, scenes_training=N)

for i in range(len(PC_scenes_training)):
    PC_pair = PC_scenes_training[i][0]
    zero_vec = np.zeros((3))
    T0 = PC_pair.pose0
    T1 = PC_pair.pose1
    viewpoint1_CS0 = torch.matmul(torch.linalg.inv(T0), T1).cpu().numpy()[:3, 3]
    PC_union = PC_pair.PCUnion.pc.cpu().numpy()

    print(f"\nScene: {i}")
    visible_inds, visible_inds_pc0, visible_inds_pc1 = covisible_inds(PC_pair.PC0, PC_pair.PCUnion, T0, T1)
    # print(f"Percentage visible points own viewpoint vs other " +
    #       f"{np.around(100*len(visible_inds)/pc_in.shape[0], 3)} % vs "+
    #       f"{np.around(100*len(visible_inds_other)/pc_in.shape[0], 3)} %")
    print(f"Percentage visible points " +
          f"{np.around(100*len(visible_inds)/PC_union.shape[0], 3)} %")

    #print(f"Percentage of point within {T_close_thresh} m {np.around(100*pc_in[np.linalg.norm(pc_in, axis=1) < T_close_thresh].shape[0]/pc_in.shape[0], 4)} %")

    plot_points(PC_union, viewpoint=zero_vec, plot_viewpoint=True, extra_viewpoint=viewpoint1_CS0, points_special_inds=visible_inds)

In [ ]:
%%timeit
visible_inds = visible_points(points, viewpoint, param_radius=3.7)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from visualization.point_e_tools import get_point_e_model, scatter_spheres
from visualization.point_e.point_e.util.plotting import plot_point_cloud

from utils.visibility import visible_points

In [ ]:
# Generate synthetic point cloud with point-E (from text to point cloud)
# Takes about 1-2 min to run so don't re-run it if not necessary
samples, sampler = get_point_e_model(text='a pink car')

In [ ]:
pc = sampler.output_to_point_clouds(samples)[0]
viewpoint = np.array([0.45, -0.4, 0.5])
visible_inds = visible_points(pc.coords, viewpoint, param_radius=3.7)
print(pc.coords.shape, type(pc.coords))
pc.coords = pc.coords[visible_inds]
print(pc.coords.shape, type(pc.coords))


In [ ]:

#pc = sampler.output_to_point_clouds(samples)[0]
size = 0.50
fig = plot_point_cloud(pc, grid_size=1, fixed_bounds=((-size, -size, -size),(size, size, size)), grid_1d=True)
# Get the current Axes3D object
ax = plt.gca()
# Reduce the opacity of the point cloud
axes = fig.get_axes()
for ax in axes:
    for coll in ax.collections:
        coll.set_alpha(0.1)  # Set the alpha to 0.5 or any other value less than 1



# Add dotted lines from the point to the respective axes
#ax = scatter_spheres(x=-0.3, y=-0.4, z=0.3, ax=ax, col="r", size=size)
ax = scatter_spheres(x=0.45, y=-0.4, z=0.5, ax=ax, col="b", size=size)
plt.show()